# Parte 2: Feature Engineering y Cálculo de KPIs Operativos

Este notebook automatiza el procesamiento analítico de la telemetría de la flota Volvo (JRC).
A partir del dataset limpio de la fase anterior, realizaremos una extracción quirúrgica de variables complejas (JSON) y calcularemos los diferenciales temporales ($\Delta$) necesarios para construir los KPIs core de negocio.

## Estructura del Proceso:
1. **Setup de Entorno**: Carga segura de los datos y validación de integridad.
2. **Parsing de Telemetría JSON**: Extracción directa y estandarizada de métricas embebidas.
3. **Cálculo de deltas ($\Delta$)**: Procesamiento de contadores de vida útil agrupados por unidad.
4. **Construcción de KPIs**: Rendimiento e indicadores de eficiencia operativa.

In [ ]:
# ==============================================================================
# 1. SETUP DE ENTORNO Y CARGA DE DATOS
# ==============================================================================

import json
import numpy as np
import pandas as pd

# Definición de constantes globales (Factores de conversión estandarizados)
ML_TO_GAL = 3785.41  # Mililitros a Galones Americanos
M_TO_KM = 1000.0     # Metros a Kilómetros
SEC_TO_H = 3600.0    # Segundos a Horas

# Configuración de visualización defensiva en Pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

print("[INFO] Cargando datasets a la memoria (con protección contra errores de formato)...")

try:
    # Cargamos ambos datasets con tolerancia a errores de JSON (lineterminator)
    df_full = pd.read_csv('dataset_analitico.csv', low_memory=False, lineterminator='\n', on_bad_lines='warn')
    df_acc = pd.read_csv('dataset_acumulado.csv', low_memory=False, lineterminator='\n', on_bad_lines='warn')

    print(f"[SUCCESS] Maestro analítico (Rendimiento): {df_full.shape[0]:,} filas × {df_full.shape[1]} columnas.")
    print(f"[SUCCESS] Maestro acumulado (JSONs):       {df_acc.shape[0]:,} filas × {df_acc.shape[1]} columnas.")

except Exception as e:
    print(f"[CRITICAL ERROR] Error al leer los datos: {e}")

[INFO] Cargando datasets a la memoria (con protección contra errores de formato)...
[SUCCESS] Maestro analítico (Rendimiento): 110,117 filas × 102 columnas.
[SUCCESS] Maestro acumulado (JSONs):       12,076 filas × 102 columnas.


## 2. Motor de Extracción Quirúrgica para Objetos JSON

Las columnas de comportamiento mecánico (Zona Verde, Sobrecarga, Exceso de velocidad) vienen empaquetadas desde los servidores de Volvo en estructuras de diccionarios (JSON).

Para optimizar el uso de memoria RAM y evitar sub-procesos lentos, implementamos una función vectorizada con manejo de excepciones tolerante a fallos (`try-except`). Esta función "abre" la celda, extrae la métrica requerida (`seconds`) y le aplica el factor de conversión en un solo paso matemático.

In [ ]:
# ==============================================================================
# 2. MOTOR DE PARSING JSON Y EXTRACCIÓN (dataset_acumulado)
# ==============================================================================

def extract_telemetry_metric(cell, target_key, conversion_factor=1.0):
    """Saca quirúrgicamente una sub-clave de una celda JSON y la convierte a
    la unidad destino. Devuelve NaN si la celda está corrupta o vacía.
    """
    if pd.isna(cell):
        return np.nan

    try:
        # Reemplazo defensivo para asegurar formato JSON válido
        if isinstance(cell, str):
            cleaned_cell = cell.replace("'", '"')
            payload = json.loads(cleaned_cell)
        else:
            payload = cell

        if isinstance(payload, dict):
            raw_value = payload.get(target_key)
            if raw_value is not None:
                return float(raw_value) / conversion_factor
        return np.nan

    except (json.JSONDecodeError, ValueError, TypeError):
        return np.nan

print("[INFO] Extrayendo tiempos operativos desde bloques JSON en df_acc...")

# Extraemos solo lo necesario aplicando el factor de Segundos a Horas
df_acc["tiempo_zona_verde_h"] = df_acc["tiempo_en_zona_verde_rpm"].apply(
    lambda c: extract_telemetry_metric(c, "seconds", SEC_TO_H)
)
df_acc["tiempo_fuera_zona_verde_h"] = df_acc["tiempo_fuera_zona_verde_rpm"].apply(
    lambda c: extract_telemetry_metric(c, "seconds", SEC_TO_H)
)
df_acc["tiempo_sobre_revolucion_h"] = df_acc["tiempo_sobre_revolucion_motor"].apply(
    lambda c: extract_telemetry_metric(c, "seconds", SEC_TO_H)
)
df_acc["tiempo_sobrecarga_motor_h"] = df_acc["tiempo_sobrecarga_motor"].apply(
    lambda c: extract_telemetry_metric(c, "seconds", SEC_TO_H)
)
df_acc["tiempo_exceso_velocidad_h"] = df_acc["tiempo_exceso_velocidad"].apply(
    lambda c: extract_telemetry_metric(c, "seconds", SEC_TO_H)
)

print("[SUCCESS] Extracción completada.")


# ==============================================================================
# 2.1. CÁLCULO DE DELTAS EN EL DATASET ACUMULADO (Hábitos de Conducción)
# ==============================================================================

print("[INFO] Calculando diferenciales (Deltas) para los hábitos de conducción...")

# 1. Asegurar orden cronológico para no restar al revés
df_acc = df_acc.sort_values(['id_vehiculo', 'fecha_creacion_evento'])

# 2. Lista de las columnas de horas que acabamos de extraer de los JSON
columnas_horas_json = [
    "tiempo_zona_verde_h",
    "tiempo_fuera_zona_verde_h",
    "tiempo_sobre_revolucion_h",
    "tiempo_sobrecarga_motor_h",
    "tiempo_exceso_velocidad_h"
]

# 3. Aplicar el diferencial (.diff) a las horas de conducción
for col in columnas_horas_json:
    nombre_delta = f"delta_{col}"
    # Restamos para obtener el abuso exacto en ese viaje y evitamos números negativos
    df_acc[nombre_delta] = df_acc.groupby('id_vehiculo')[col].diff().fillna(0).clip(lower=0)

print("[SUCCESS] Deltas de conducción listos para el dashboard.")

[INFO] Extrayendo tiempos operativos desde bloques JSON en df_acc...
[SUCCESS] Extracción completada.
[INFO] Calculando diferenciales (Deltas) para los hábitos de conducción...
[SUCCESS] Deltas de conducción listos para el dashboard.


## 3. Cálculo de Diferenciales Temporales ($\Delta$)

Los datos de telemetría minera registran valores históricos continuos (contadores de vida útil). Para evaluar el desempeño de un turno o evento específico, es matemáticamente obligatorio calcular el delta ($\Delta$) o la diferencia entre la lectura actual y la lectura inmediatamente anterior.

**Variables transformadas:**
* Odómetro total (km) $\rightarrow$ Distancia recorrida en el evento ($\Delta$ km).
* Combustible total (gal) $\rightarrow$ Combustible consumido en el evento ($\Delta$ gal).

> **Nota Técnica:** Se utiliza `groupby('id_vehiculo')` antes de aplicar `.diff()` para evitar que el cálculo reste el odómetro inicial de un camión con el odómetro final de otro camión distinto. Además, se filtran valores negativos anómalos causados por reinicios de sensores.

In [ ]:
# ==============================================================================
# 3. CÁLCULO DE DELTAS EN EL DATASET MAESTRO (df_full)
# ==============================================================================

import numpy as np

print("[INFO] Calculando diferenciales (Deltas) de consumo y distancia...")

# 1. Asegurar orden cronológico estricto por seguridad
df_full = df_full.sort_values(['id_vehiculo', 'fecha_creacion_evento'])

# 2. Calcular la diferencia con la fila anterior agrupando por camión
df_full['delta_odometro_km'] = df_full.groupby('id_vehiculo')['odometro_total_km'].diff()
df_full['delta_combustible_gal'] = df_full.groupby('id_vehiculo')['combustible_total_gal'].diff()

# 3. Limpieza de deltas:
# - La primera fila de cada camión dará NaN (no hay anterior), lo llenamos con 0.
# - Si un sensor se reinició y dio negativo, lo limitamos a 0 usando .clip(lower=0)
df_full['delta_odometro_km'] = df_full['delta_odometro_km'].fillna(0).clip(lower=0)
df_full['delta_combustible_gal'] = df_full['delta_combustible_gal'].fillna(0).clip(lower=0)

print("[SUCCESS] Deltas calculados correctamente.")



[INFO] Calculando diferenciales (Deltas) de consumo y distancia...
[SUCCESS] Deltas calculados correctamente.


## 4. Creación de KPIs de Negocio (Business Intelligence)

En esta sección, transformamos las variables operativas base (Deltas y horas) en **Indicadores Clave de Rendimiento (KPIs)**. Estos indicadores son las métricas definitivas que el equipo de visualización (Grupo C) consumirá en el dashboard para evaluar la eficiencia de la flota y tomar decisiones gerenciales.

Se han diseñado tres KPIs fundamentales aplicando vectorización lógica para procesar los eventos a alta velocidad:

1. **Rendimiento Operativo ($gal/km$):**
   Mide la eficiencia de combustible aislando la distancia real recorrida en cada tramo. Se implementó una protección matemática (`np.where`) para evitar errores fatales de "división por cero" en los registros donde el camión consumió combustible pero su odómetro no avanzó.

2. **Utilización Efectiva (%):**
   Evalúa qué porcentaje del tiempo que el motor estuvo encendido se usó realmente para desplazarse. Esta métrica penaliza automáticamente el tiempo en ralentí (motor encendido pero vehículo detenido), evidenciando las ineficiencias logísticas en los puntos de carguío y descarga.

3. **Banderas de Alerta Crítica (Tablero):**
   Se tipificaron los estados categóricos del sistema Volvo (`RED`, `YELLOW`) transformándolos en variables binarias (`1` o `0`). Al convertir alertas de texto en *flags* numéricos, permitimos que el frontend filtre, cuente y cruce fallas críticas mecánicas de manera instantánea.

In [ ]:
# ==============================================================================
# 4. CREACIÓN DE KPIs DE NEGOCIO (Business Intelligence)
# ==============================================================================

print("[INFO] Generando KPIs operativos para el Grupo C...")

# KPI 1: Rendimiento Operativo (Galones por Kilómetro)
# Usamos np.where para evitar el error de división por cero cuando el camión no se movió
df_full['kpi_rendimiento_gal_km'] = np.where(
    df_full['delta_odometro_km'] > 0,
    df_full['delta_combustible_gal'] / df_full['delta_odometro_km'],
    np.nan # Si no avanzó, no hay rendimiento calculable
)

# KPI 2: Porcentaje de Utilización Efectiva (En el momento del evento)
# Fórmula: Tiempo en movimiento / (Tiempo detenido con motor encendido + Tiempo en movimiento)
tiempo_total_operacion = df_full['tiempo_en_movimiento_h'] + df_full['tiempo_motor_encendido_detenido_h']

df_full['kpi_utilizacion_efectiva_pct'] = np.where(
    tiempo_total_operacion > 0,
    (df_full['tiempo_en_movimiento_h'] / tiempo_total_operacion) * 100,
    np.nan
)

# KPI 3: Bandera (Flag) de Alerta Crítica del Tablero (Corregido y Tipado)
# Marcamos estrictamente 'RED' como alerta crítica y 'YELLOW' como advertencia opcional
if 'estado_alerta_tablero' in df_full.columns:
    df_full['flag_alerta_critica'] = np.where(df_full['estado_alerta_tablero'] == 'RED', 1, 0)

    df_full['flag_alerta_preventiva'] = np.where(df_full['estado_alerta_tablero'] == 'YELLOW', 1, 0)

print("[SUCCESS] KPIs base generados exitosamente.")

# Mostramos una muestra aleatoria para validar los cálculos (solo las columnas nuevas)
columnas_validacion = [
    'id_vehiculo', 'delta_odometro_km', 'delta_combustible_gal',
    'kpi_rendimiento_gal_km', 'kpi_utilizacion_efectiva_pct'
]
display(df_full[columnas_validacion].sample(20))

[INFO] Generando KPIs operativos para el Grupo C...
[SUCCESS] KPIs base generados exitosamente.


,id_vehiculo,delta_odometro_km,delta_combustible_gal,kpi_rendimiento_gal_km,kpi_utilizacion_efectiva_pct
27848,93KXG30D4TE625371,0.2150,0.0660,0.3072,NaN
99072,93KXG40G9SE616577,0.2800,0.1162,0.4151,NaN
6836,93KXG30D1TE625370,0.0000,0.0026,NaN,67.6880
5616,93KXG30D1TE625370,0.0000,0.0000,NaN,NaN
62345,93KXG40G0RE941880,0.1750,0.0476,0.2717,NaN
58296,93KXG30DXTE625616,0.0000,0.0000,NaN,67.6972
76623,93KXG40G5SE616052,0.1150,0.0238,0.2067,NaN
17722,93KXG30D3TE624343,0.1750,0.1242,0.7095,NaN
67150,93KXG40G3SE616580,12.7100,0.0000,0.0000,NaN
83268,93KXG40G6RE941877,0.4300,0.2378,0.5529,NaN


## 5. Detección Estadística de Anomalías (Outliers)

Para identificar eventos de consumo de combustible anormalmente altos o bajos sin depender de modelos predictivos, implementamos el método robusto del **Rango Intercuartílico (IQR)** sobre el KPI de rendimiento ($gal/km$).

Este algoritmo establece barreras matemáticas dinámicas basadas en la distribución real de la flota de JRC, etiquetando con un `1` (True) cualquier evento que exceda el límite superior de consumo atípico.

In [ ]:
# ==============================================================================
# 5. DETECCIÓN DE OUTLIERS DE CONSUMO (MÉTODO IQR)
# ==============================================================================

print("[INFO] Ejecutando algoritmo de detección de anomalías (IQR)...")

# 1. Aislar la columna de rendimiento, ignorando los NaNs para no romper la matemática
rendimiento_limpio = df_full['kpi_rendimiento_gal_km'].dropna()

# 2. Calcular los Cuartiles y el IQR
Q1 = rendimiento_limpio.quantile(0.25)
Q3 = rendimiento_limpio.quantile(0.75)
IQR = Q3 - Q1

# 3. Definir las fronteras matemáticas
# Usamos el multiplicador estándar de la estadística (1.5)
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"  -> Límite Inferior (Consumo inusualmente bajo): {limite_inferior:.4f} gal/km")
print(f"  -> Límite Superior (Consumo atípico/crítico):   {limite_superior:.4f} gal/km")

# 4. Crear la variable "Bandera" (Flag) en el Dataset
# np.where asigna 1 si es anomalía, 0 si es un evento normal
df_full['flag_outlier_consumo'] = np.where(
    (df_full['kpi_rendimiento_gal_km'] > limite_superior) |
    (df_full['kpi_rendimiento_gal_km'] < limite_inferior),
    1,
    0
)

# 5. Reporte rápido de resultados
total_eventos = df_full['kpi_rendimiento_gal_km'].notna().sum()
total_anomalias = df_full['flag_outlier_consumo'].sum()
porcentaje = (total_anomalias / total_eventos) * 100 if total_eventos > 0 else 0

print(f"[SUCCESS] Detección finalizada.")
print(f"  -> Se analizaron {total_eventos:,} eventos válidos.")
print(f"  -> Se detectaron {total_anomalias:,} anomalías ({porcentaje:.2f}% de la operación).")

# Mostrar una muestra de los camiones infractores (los que sacaron 1)
print("\nMuestra de eventos anómalos detectados:")
display(df_full[df_full['flag_outlier_consumo'] == 1][
    ['id_vehiculo', 'fecha_creacion_evento', 'kpi_rendimiento_gal_km', 'flag_outlier_consumo']
].head())

[INFO] Ejecutando algoritmo de detección de anomalías (IQR)...
  -> Límite Inferior (Consumo inusualmente bajo): -0.6845 gal/km
  -> Límite Superior (Consumo atípico/crítico):   1.4976 gal/km
[SUCCESS] Detección finalizada.
  -> Se analizaron 82,408 eventos válidos.
  -> Se detectaron 2,997 anomalías (3.64% de la operación).

Muestra de eventos anómalos detectados:


,id_vehiculo,fecha_creacion_evento,kpi_rendimiento_gal_km,flag_outlier_consumo
166,93KXG30D0TE625539,2026-03-24 21:01:26+00:00,3.1701,1
203,93KXG30D0TE625539,2026-03-27 01:00:51+00:00,57734.6602,1
279,93KXG30D0TE625539,2026-03-30 01:05:37+00:00,11.6236,1
347,93KXG30D0TE625539,2026-03-31 11:00:00+00:00,3.5663,1
381,93KXG30D0TE625539,2026-04-01 13:00:00+00:00,1.7024,1


## 6. Análisis de Correlación (Pearson)

Para responder a la pregunta de negocio: *"¿Las alertas críticas del tablero están relacionadas con el exceso de consumo de combustible?"*, aplicamos una matriz de **Correlación de Pearson**.

Este análisis cruzará nuestras variables de rendimiento (`kpi_rendimiento_gal_km`, `flag_outlier_consumo`) con las banderas de alerta y los tiempos de operación, arrojando un valor entre -1 y 1. Un valor cercano a 1 indicará que cuando una variable sube, la otra también (ej. más alertas = más consumo).

In [ ]:
# ==============================================================================
# 6. MATRIZ DE CORRELACIÓN DE VARIABLES CRÍTICAS
# ==============================================================================

print("[INFO] Generando matriz de correlación de Pearson...")

# Seleccionamos estrictamente las columnas numéricas y los KPIs que nos importan
columnas_correlacion = [
    'delta_odometro_km',
    'delta_combustible_gal',
    'kpi_rendimiento_gal_km',
    'kpi_utilizacion_efectiva_pct',
    'flag_outlier_consumo'
]

# Añadimos la alerta crítica solo si se logró crear en pasos anteriores
if 'flag_alerta_critica' in df_full.columns:
    columnas_correlacion.append('flag_alerta_critica')

# Generamos la matriz matemática
matriz_corr = df_full[columnas_correlacion].corr(method='pearson')

print("[SUCCESS] Correlación calculada. Resultados principales respecto al rendimiento:")

# Filtramos solo la columna de interés para no saturar la pantalla
if not matriz_corr.empty:
    correlacion_rendimiento = matriz_corr[['kpi_rendimiento_gal_km', 'flag_outlier_consumo']].round(4)
    display(correlacion_rendimiento)
else:
    print("[WARNING] No se pudo generar la correlación por falta de datos numéricos válidos.")

[INFO] Generando matriz de correlación de Pearson...
[SUCCESS] Correlación calculada. Resultados principales respecto al rendimiento:


,kpi_rendimiento_gal_km,flag_outlier_consumo
delta_odometro_km,-0.0030,-0.0087
delta_combustible_gal,0.4129,0.1612
kpi_rendimiento_gal_km,1.0000,0.1152
kpi_utilizacion_efectiva_pct,0.0011,0.0147
flag_outlier_consumo,0.1152,1.0000
flag_alerta_critica,-0.0004,-0.0039


## 7. Exportación de Tablas Maestras (Handoff a Grupo C)

El proceso de *Feature Engineering* ha concluido. Los datasets originales han sido transformados, limpiados y enriquecidos con métricas de negocio.

Se procede a exportar los resultados en archivos CSV planos, los cuales serán la fuente de datos directa para la construcción del Dashboard en PowerBI / Streamlit.

In [ ]:
# ==============================================================================
# 7. EXPORTACIÓN FINAL DE LOS DATASETS
# ==============================================================================

print("[INFO] Preparando exportación de los datasets maestros...")

# Nombres de los archivos finales (convención clara para el Grupo C)
OUT_FULL_MASTER = 'dataset_master_kpis_analitico.csv'
OUT_ACC_MASTER = 'dataset_master_kpis_acumulado.csv'

try:
    # Exportamos el archivo analítico (Rendimiento, Rutas, KPIs)
    df_full.to_csv(OUT_FULL_MASTER, index=False, encoding='utf-8-sig')
    print(f"[SUCCESS] Exportado: {OUT_FULL_MASTER} ({df_full.shape[0]:,} filas)")

    # Exportamos el archivo acumulado (Hábitos de choferes desempaquetados de JSON)
    df_acc.to_csv(OUT_ACC_MASTER, index=False, encoding='utf-8-sig')
    print(f"[SUCCESS] Exportado: {OUT_ACC_MASTER} ({df_acc.shape[0]:,} filas)")

    print("\n" + "="*60)
    print("EL PIPELINE DE FEATURE ENGINEERING HA FINALIZADO CON ÉXITO")
    print("="*60)
    print("Archivos listos para entregar al Grupo C.")

except Exception as e:
    print(f"[CRITICAL ERROR] Fallo al exportar los archivos: {e}")

[INFO] Preparando exportación de los datasets maestros...
[SUCCESS] Exportado: dataset_master_kpis_analitico.csv (110,117 filas)
[SUCCESS] Exportado: dataset_master_kpis_acumulado.csv (12,076 filas)

EL PIPELINE DE FEATURE ENGINEERING HA FINALIZADO CON ÉXITO
Archivos listos para entregar al Grupo C.
